In [1]:
print("Test")

Test


In [2]:
# Imports
import sys
import pathlib

# Add the project's root directory to the Python path
sys.path.append(pathlib.Path("../").resolve().as_posix())

# Configurations
seed = 42

# Paths
DATA_DIR = pathlib.Path("../data/")
ENC2017_ROOT = DATA_DIR / "enc2017"
UD_ET_EDT_ROOT = DATA_DIR / "ud_et_edt"
HOMONYMS_ROOT = DATA_DIR / "homonymous_word_forms"

ENC2017_DIRS = {
    "processed": ENC2017_ROOT / "processed",
    "raw": ENC2017_ROOT / "raw",
}

UD_ET_EDT_DIRS = {
    "processed": UD_ET_EDT_ROOT / "processed",
    "raw": UD_ET_EDT_ROOT / "raw",
}

HOMONYMS_DIRS = {
    "processed": HOMONYMS_ROOT / "processed",
    "annotations": HOMONYMS_ROOT / "annotations",
}

OUTPUT_DIR = pathlib.Path("../outputs/")

MODEL_DIR = pathlib.Path("../models/")

In [3]:
import pandas as pd
import ast
import tqdm
import glob
import estnltk
from estnltk.default_resolver import make_resolver
from estnltk.taggers import VabamorfAnalyzer
from estnltk_neural.taggers import StanzaSyntaxTagger

e:\Git_projects\EstNLTK\simpletransformers\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# Download the Stanza models for Estonian if not already present
stanza_syntax_tagger = StanzaSyntaxTagger(
    input_type="morph_analysis",
    input_morph_layer="morph_analysis",
    add_parent_and_children=True,
)
# estnltk.download('stanzasyntaxtagger')

In [7]:
# from estnltk.taggers import StanzaSyntaxWebTagger
# stanza_syntax_web_tagger = \
#     StanzaSyntaxWebTagger(url='https://api.tartunlp.ai/estnltk/tagger/stanza_syntax')
# stanza_syntax_web_tagger

In [5]:
sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

Text(text='Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav.')

In [6]:
display(text_obj.sentences[0].stanza_syntax)

Layer(name='stanza_syntax', attributes=('id', 'lemma', 'upostag', 'xpostag', 'feats', 'head', 'deprel', 'deps', 'misc', 'parent_span', 'children'), spans=SL[Span('Ta', [{'id': 1, 'lemma': 'tema', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('n', 'n')]), 'head': 2, 'deprel': 'nsubj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('andis', [{'id': 2, 'lemma': 'andma', 'upostag': 'V', 'xpostag': 'V', 'feats': OrderedDict([('s', 's')]), 'head': 0, 'deprel': 'root', 'deps': '_', 'misc': '_', 'parent_span': None, 'children': <class 'tuple'>}]),
Span('lendurist', [{'id': 3, 'lemma': 'lendur', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('el', 'el')]), 'head': 4, 'deprel': 'nmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('abikaasale', [{'id': 4, 'lemma': 'abikaasa', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('all', 'all')]), 'head': 2, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('oma', [{'id': 5, 'lemma': 'oma', 'upostag': 'P', 'xpostag': 'P', 'feats': OrderedDict([('sg', 'sg'), ('g', 'g')]), 'head': 6, 'deprel': 'nmod', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}]),
Span('raamatu', [{'id': 6, 'lemma': 'raamat', 'upostag': 'S', 'xpostag': 'S', 'feats': OrderedDict([('sg', 'sg'), ('g', 'g')]), 'head': 2, 'deprel': 'obj', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': <class 'tuple'>}]),
Span('.', [{'id': 7, 'lemma': '.', 'upostag': 'Z', 'xpostag': 'Z', 'feats': OrderedDict(), 'head': 2, 'deprel': 'punct', 'deps': '_', 'misc': '_', 'parent_span': <class 'estnltk_core.layer.span.Span'>, 'children': ()}])])

In [ ]:
for ss in text_obj.stanza_syntax:
    if ss.children and (ss.upostag not in ["V"] or ss.xpostag not in ["V"]):
        for child in ss.children:
            if child.deprel == "nmod" and (
                child.upostag == "S" or child.xpostag == "S"
            ):
                print(f"{ss.text} -> {child.text} (nsubj)")
                print(f"Phrase: {ss.text} {child.text}")

abikaasale -> Lendurist (nsubj)
Phrase: abikaasale Lendurist


In [7]:
for ann in text_obj.stanza_syntax:
    print(ann.head)
    print(type(ann))

2
<class 'estnltk_core.layer.span.Span'>
0
<class 'estnltk_core.layer.span.Span'>
4
<class 'estnltk_core.layer.span.Span'>
2
<class 'estnltk_core.layer.span.Span'>
6
<class 'estnltk_core.layer.span.Span'>
2
<class 'estnltk_core.layer.span.Span'>
2
<class 'estnltk_core.layer.span.Span'>
2
<class 'estnltk_core.layer.span.Span'>
5
<class 'estnltk_core.layer.span.Span'>
5
<class 'estnltk_core.layer.span.Span'>
5
<class 'estnltk_core.layer.span.Span'>
0
<class 'estnltk_core.layer.span.Span'>
5
<class 'estnltk_core.layer.span.Span'>


In [17]:
stanza_ann = text_obj.sentences[0].stanza_syntax
for ann in stanza_ann:
    print(getattr(ann, "deprel", None))

nsubj
root
nmod
obj
nmod
obj
punct


**Recommended scaffold**

- `SyntaxGraphIndex`
  - Responsibility: convert one sentence’s syntax layer into a reusable graph view.
  - Stores:
    - node lookup by token id
    - parent lookup
    - children lookup
    - optional cached token attributes for fast filtering
  - Why you need it: this avoids rescanning the whole layer repeatedly.

- `NodeConstraint`
  - Responsibility: represent conditions on a single node.
  - Should support:
    - exact match
    - negation
    - wildcard/no constraint
    - subset-style feature matching for `feats`
  - Important: keep this explicit and declarative. I would avoid encoding these as plain nested string lists if readability matters.

- `EdgeConstraint`
  - Responsibility: represent one directed arc in the dependency chain.
  - Should include:
    - direction
    - `deprel` condition
    - optional structural constraints such as “must be direct child” or “must be ancestor within N hops”
  - Why: dependency patterns are about relations, not just node attributes.

- `PathPattern`
  - Responsibility: define the whole chain pattern.
  - Contains:
    - ordered steps
    - node constraints for each step
    - edge constraints between steps
  - This is the object you want to make readable for future you.
  - Example idea in words: “step 0 is a noun in elative case; step 1 is its head noun in allative case; the arc is `nmod`”.

- `ChainMatch`
  - Responsibility: store one successful match in a standard form.
  - Should include:
    - the role name for each matched node, such as `source`, `target`, `pivot`
    - token ids
    - spans or text spans
    - the traversed arcs
    - the attributes that were checked
  - This is the contract between matcher and decorator.

- `DepChainMatcher`
  - Responsibility: find all matches for one `PathPattern` over one sentence.
  - Behaviour:
    - choose anchors from the most selective step
    - traverse directionally
    - prune immediately when constraints fail
    - emit one `ChainMatch` per valid path
  - For your current use case, this is the core engine.

- `PhraseDecorator`
  - Responsibility: convert a `ChainMatch` into output annotations.
  - Likely output:
    - the phrase text
    - the copied syntax row for the matched word(s)
    - any extra labels you want downstream
  - If you want to “copy the syntax layer row to the matching word”, this belongs here, not in the matcher.

- `DepChainTagger`
  - Responsibility: tie everything together as an EstNLTK-style tagger.
  - Flow:
    - receive `Text`
    - read syntax layer
    - build graph index
    - run matcher
    - decorate matches into relational layer output

**Algorithm direction**

For your sample phrase, I would use this process:

1. Build the graph index from the syntax layer once.
2. Select candidate anchors using the most restrictive node condition, usually the dependent noun with the strongest morphological filter.
3. For each anchor, walk only the required direction through the dependency graph.
4. At each step, test:
   - edge label
   - direction
   - node PoS
   - feature constraints
5. If all steps succeed, emit a match immediately.
6. Let the decorator decide how that match is represented in the relational layer.

This is more readable than a full BFS over the tree because you already know the chain length is small and the rules are selective.

**How I would frame the rule syntax**

You said you want exact, negation, and wildcard support, which is sensible. I would keep that, but I would not express all logic only as nested lists of strings. That becomes hard to read once chains grow.

A better mental model is:

- each step = one node specification
- each link = one arc specification
- each node specification = a small set of named attribute tests
- each attribute test = exact, negated exact, subset, or wildcard

That gives you readability without forcing a mini regex language onto graph patterns.

That separation is the right balance for your goal. It gives you readable rules, supports directional dependency chains, and keeps the relational layer generation clean.

**1. SyntaxGraphIndex**
Purpose: sentence-level dependency graph cache for fast traversal.

Fields:

- syntax_layer: original stanza_syntax layer object
- nodes_by_id: map from token id to token annotation row
- parent_by_id: map from token id to parent id (or null/root)
- children_by_id: map from token id to list of child ids
- token_order: list of token ids in sentence order
- sent_id or sentence_span: optional sentence metadata
- lookup_cache: optional cache for repeated attribute access

Methods:

- build_from_layer(syntax_layer): populates all index maps
- get_node(token_id): returns annotation row
- get_parent(token_id): returns parent node or null
- get_children(token_id): returns child nodes list
- iter_nodes(): yields nodes in sentence order
- iter_edges(direction): yields directed edges
- get_root_nodes(): returns roots
- has_node(token_id): existence check
- validate_tree(): basic diagnostics (cycles, missing heads, orphan references)

**2. ValueCondition**
Purpose: reusable matcher for one scalar attribute.

Fields:

- mode: one of exact, not_exact, wildcard
- value: expected value for exact or not_exact
- allow_missing: whether missing attribute passes
- normaliser: optional normalisation rule (case-folding, mapping tags)

Methods:

- matches(actual_value): true or false
- describe(): human-readable condition text
- validate(): checks config consistency

**3. FeatureCondition**
Purpose: matcher for feats dictionary style attributes.

Fields:

- mode: one of exact_set, subset, negation, wildcard
- required: key-value constraints that must hold
- forbidden: key-value constraints that must not hold
- allow_extra_keys: relevant for exact_set vs subset
- allow_missing: whether absent feats pass

Methods:

- matches(feats_dict): true or false
- describe(): readable summary
- validate(): checks impossible combinations

**4. NodeConstraint**
Purpose: all constraints for one node in the chain.

Fields:

- role: semantic role label for this step (source, pivot, target)
- upostag_condition: ValueCondition
- xpostag_condition: ValueCondition
- lemma_condition: ValueCondition
- deprel_at_node_condition: optional ValueCondition for node’s own deprel
- feats_condition: FeatureCondition
- extra_predicates: optional list of advanced checks
- optional: whether this node step may be skipped

Methods:

- matches(node_annotation): true or false
- score_selectivity(): heuristic for anchor selection
- describe(): readable node rule
- validate(): config validation

**5. EdgeConstraint**
Purpose: directional relation rule between two adjacent steps.

Fields:

- direction: up, down, or either
- deprel_condition: ValueCondition (usually exact or set-like exact options)
- min_hops: usually 1
- max_hops: usually 1 for your current task
- allow_crossing_sentence: usually false
- optional: whether edge may be absent for optional steps

Methods:

- matches(edge_context): true or false
- enumerate_next_nodes(index, current_id): yields candidates by direction and hop bounds
- describe(): readable edge rule
- validate(): config checks

**6. PathPattern**
Purpose: full directional dependency chain definition.

Fields:

- name: pattern name
- version: rule schema version
- node_steps: ordered list of NodeConstraint
- edge_steps: ordered list of EdgeConstraint (count = node_steps - 1)
- anchor_role: role used to start matching
- emit_roles: roles to include in output phrase
- metadata: tags, notes, provenance
- priority: for conflict handling if needed later

Methods:

- validate(): structure checks
- choose_anchor_role(): picks most selective role if not set
- describe(): compact human-readable pattern
- to_dict and from_dict: serialisation

**7. ChainMatch**
Purpose: canonical output of the matcher, consumed by decorators.

Fields:

- pattern_name: which pattern matched
- sentence_ref: sentence identity or span
- role_to_token_id: map role to matched token id
- role_to_node_snapshot: copied annotation row per role
- traversed_edges: ordered list of realised edges
- matched_text: optional reconstructed phrase text
- evidence: optional diagnostics of passed constraints
- confidence: optional score
- match_id: stable identifier
- metadata: extra annotations

Methods:

- get_role_node(role): node snapshot for a role
- get_phrase_tokens(order): ordered tokens for phrase rendering
- to_relational_records(): converts to one-or-more relational layer entries
- to_dict(): serialisable representation

**8. MatchCollector**
Purpose: accumulate all valid matches, including ambiguity handling.

Fields:

- matches: list of ChainMatch
- dedup_policy: none, by span, by role tuple
- include_ambiguous: true for your requirement
- stats: counters for diagnostics

Methods:

- add(match): appends according to policy
- extend(match_list): bulk add
- deduplicate(): optional post-step
- summary(): counts per pattern, sentence, and role

**9. DepChainMatcher**
Purpose: executes patterns over one sentence index.

Fields:

- pattern: PathPattern
- max_search_nodes: safety bound
- prune_on_first_failure: true
- traverse_strategy: bounded_dfs (recommended)
- include_partial_debug: optional

Methods:

- find_matches(index): returns list of ChainMatch
- find_from_anchor(index, anchor_token_id): search from one anchor
- expand_step(index, current_state, step_idx): traverses one edge and validates next node
- is_complete(state): completion check
- make_match(state): builds ChainMatch
- prune(state): early rejection logic
- diagnostics(): optional debug details

**10. PhraseDecorator**
Purpose: converts ChainMatch into relational layer rows for downstream taggers.

Fields:

- output_layer_name: relational layer name
- output_attributes: fields to store
- copied_syntax_fields: which syntax row fields to copy
- phrase_format: token order and join style
- semantic_layer_names: optional extra layers to consult

Methods:

- decorate(match, text_obj): returns one or more output annotations
- build_phrase_text(match, text_obj): phrase string construction
- copy_node_row(match, role): selected syntax-row copy
- attach_semantic_flags(match, text_obj): optional doer-like flags
- validate_output_schema(): consistency check

**11. DepChainTagger**
Purpose: orchestrator with EstNLTK-style API.

Fields:

- input_layers: required layers list (for example stanza_syntax and optionally morph layer)
- output_layer: relational layer name
- patterns: list of PathPattern
- matcher_factory: creates DepChainMatcher
- decorator: PhraseDecorator
- conflict_resolver: strategy for overlapping outputs
- config: run options and debug toggles

Methods:

- make_layer_template(): defines relational output structure
- make_layer(text, layers, status): full pipeline entrypoint
- run_sentence(sentence_layer): per-sentence execution
- apply_patterns(index): runs all patterns, collects matches
- resolve_conflicts(layer): optional overlap policy
- validate_configuration(): startup checks

**12. RuleValidator (optional but highly recommended)**
Purpose: fail-fast validation and readability guarantees.

Fields:

- supported_node_attributes
- supported_edge_directions
- supported_condition_modes
- strict_mode

Methods:

- validate_pattern(pattern): schema and logic checks
- validate_constraints(pattern): impossible combinations
- lint_readability(pattern): warns on overly complex patterns

**Suggested minimum first milestone**

1. Implement only SyntaxGraphIndex, ValueCondition, FeatureCondition, NodeConstraint, EdgeConstraint, PathPattern, ChainMatch, DepChainMatcher, DepChainTagger.
2. Keep one pattern: source noun with elative features, upward nmod edge, target noun with allative features.
3. Emit all valid matches separately, one-by-one, with no deduplication.
4. Add PhraseDecorator once matching is stable.

---

**Complete Constructor Parameters for All Classes**

**1. SyntaxGraphIndex** (redesigned)

```
Constructor parameters:
- sentence_span (estnltk.Span): one sentence span with stanza_syntax accessible
```

Rationale: One index per sentence. Cleaner, no ID collisions.

---

**2. ValueCondition**

```
Constructor parameters:
- mode (str): one of "exact", "negation", "wildcard"
- value (Any): the expected value (for "exact" and "negation" modes only, None for "wildcard")
- allow_missing (bool, optional, default=False): if True, absent/None attribute passes the condition
- normalizer (Callable, optional, default=None): function to preprocess actual values before comparison (e.g., case-folding, tag mapping)
```

Example uses:

- `ValueCondition("exact", "nmod")` → matches exactly "nmod"
- `ValueCondition("negation", "V")` → matches anything except "V"
- `ValueCondition("wildcard", None)` → matches any value
- `ValueCondition("exact", "S", allow_missing=True, normalizer=str.lower)` → matches "s" or "S", or missing

---

**3. FeatureCondition**

```
Constructor parameters:
- mode (str): one of "exact_set", "subset", "negation", "wildcard"
- required (Dict[str, Any], optional, default=None): key-value pairs that must all be present
- forbidden (Dict[str, Any], optional, default=None): key-value pairs that must not be present
- allow_extra_keys (bool, optional, default=False): for "exact_set" mode, whether to allow other keys
- allow_missing (bool, optional, default=False): if True, absent feats dict passes
- normalizer (Callable, optional, default=None): function to preprocess actual feats before comparison
```

Example uses:

- `FeatureCondition("exact_set", required={"sg": "sg", "el": "el"})` → feats must be exactly this (no extra keys)
- `FeatureCondition("subset", required={"sg": "sg"})` → feats must include this pair (extra keys OK)
- `FeatureCondition("negation", forbidden={"pl": "pl"})` → feats must not have plural
- `FeatureCondition("wildcard", None)` → any feats pass

---

**4. NodeConstraint**

```
Constructor parameters:
- role (str): semantic role label, e.g. "source", "target", "pivot"
- upostag_condition (ValueCondition, optional): constraint on universal PoS
- xpostag_condition (ValueCondition, optional): constraint on language-specific PoS
- lemma_condition (ValueCondition, optional): constraint on lemma
- deprel_at_node_condition (ValueCondition, optional): node's own deprel (for nodes that are not anchors)
- feats_condition (FeatureCondition, optional): constraint on morphological features
- extra_predicates (List[Callable], optional): advanced custom checks, e.g. lambda node: node.feats.get("case") in ["el", "all"]
- optional (bool, optional, default=False): if True, this step may be skipped on optional matching
```

Rationale: explicit named constraints per attribute; separates readable conditions from complex logic.

---

**5. EdgeConstraint**

```
Constructor parameters:
- direction (str): "up" (dependent→head), "down" (head→dependent), or "either"
- deprel_condition (ValueCondition): constraint on the edge's deprel
- min_hops (int, optional, default=1): minimum distance to next node
- max_hops (int, optional, default=1): maximum distance (for skipping intermediate nodes)
- allow_crossing_sentence (bool, optional, default=False): safety flag
- optional (bool, optional, default=False): if True, this edge may be missing
```

Rationale: explicit direction, bounded hops, optional for future flexibility.

---

**6. PathPattern**

```
Constructor parameters:
- name (str): human-readable pattern name, e.g. "lendurist_abikaasale"
- node_steps (List[NodeConstraint]): ordered node constraints
- edge_steps (List[EdgeConstraint]): ordered edge constraints (len = len(node_steps) - 1)
- anchor_role (str, optional, default=None): which role to start matching from; auto-selected if None
- emit_roles (List[str], optional, default=None): which roles to include in output; all if None
- version (str, optional, default="1.0"): rule format version
- metadata (Dict, optional): tags, notes, provenance
- priority (int, optional, default=0): for conflict handling
```

Validation: `len(edge_steps) == len(node_steps) - 1` must hold.

---

**7. ChainMatch**

```
Constructor parameters:
- pattern_name (str): which pattern produced this match
- sentence_span (estnltk.Span): the sentence this match is from
- role_to_token_id (Dict[str, int]): map role to matched token id
- role_to_node_snapshot (Dict[str, Dict]): copied annotation attributes per role
- traversed_edges (List[Tuple]): ordered list of (source_id, target_id, deprel, direction)
- matched_text (str, optional): reconstructed phrase text
- evidence (Dict, optional): diagnostics of passed constraints
- confidence (float, optional, default=1.0): match score
- match_id (str, optional): stable identifier
- metadata (Dict, optional): extra annotations
```

Rationale: stable contract between matcher and decorator.

---

**8. MatchCollector**

```
Constructor parameters:
- dedup_policy (str, optional, default="none"): "none", "by_span", or "by_role_tuple"
- include_ambiguous (bool, optional, default=True): keep all matches including duplicates
- max_matches_per_sentence (int, optional, default=None): safety cap
```

---

**9. DepChainMatcher**

```
Constructor parameters:
- pattern (PathPattern): the pattern to match
- max_search_nodes (int, optional, default=100): safety bound to prevent infinite loops
- prune_on_first_failure (bool, optional, default=True): fail immediately if one constraint fails
- traverse_strategy (str, optional, default="bounded_dfs"): "bounded_dfs" or "bfs"
- debug_mode (bool, optional, default=False): enable detailed diagnostics
```

---

**10. PhraseDecorator**

```
Constructor parameters:
- output_layer_name (str): name for relational layer output
- output_attributes (List[str]): fields to store in each annotation
- copied_syntax_fields (List[str], optional, default=None): which syntax row fields to copy
- phrase_format (str, optional, default="space_joined"): how to join matched tokens
- semantic_layer_names (List[str], optional, default=None): optional extra layers to consult
```

---

**11. DepChainTagger**

```
Constructor parameters:
- patterns (List[PathPattern]): all patterns to apply
- output_layer (str): relational layer name
- input_layers (List[str], optional, default=["stanza_syntax"]): required input layers
- matcher_factory (Callable, optional, default=DepChainMatcher): custom matcher class
- decorator (PhraseDecorator): decorator instance
- conflict_resolver (str, optional, default="keep_all"): overlap policy
- config (Dict, optional): run options and debug toggles
```

---

**Summary**
The key design principle is **one parameter per concept**: don't cram multiple concerns into one parameter. This keeps rules readable and separates concerns cleanly.


In [12]:
import importlib

from scripts.DepChainTagger import SyntaxGraphIndex, ValueCondition

# importlib.reload(SyntaxGraphIndex)

In [13]:
sample_text = "Ta andis lendurist abikaasale oma raamatu. See raamat on väga huvitav."
text_obj = estnltk.Text(sample_text)
text_obj.tag_layer("morph_extended")
# stanza_syntax_web_tagger.tag(text_obj)
stanza_syntax_tagger.tag(text_obj)

# Check the SyntaxGraphIndex implementation with the sample text
for id, sentence in enumerate(text_obj.sentences):
    graph_index = SyntaxGraphIndex(
        sentence.stanza_syntax,
        sentence_id=id,
        sentence_span=(sentence.start, sentence.end),
    )
    print(
        f"Sentence ID: {graph_index.sent_id}, Sentence Span: {graph_index.sentence_span}, Token Order: {graph_index.token_order}"
    )
    print("Nodes in the graph:")
    for node in graph_index.iter_nodes():
        print(
            f"Node ID: {node.id}, Text: '{node.text}', UPOS: {node.upostag}, XPOS: {node.xpostag}, Deprel: {node.deprel}, Head: {node.head}"
        )
    print("\nEdges in the graph:")
    for node, parent, direction in graph_index.iter_edges():
        parent_text = parent.text if parent else "None"
        parent_id = parent.id if parent else "None"
        if node and parent:
            print(
                f"Node ID: {node.id} ('{node.text}') --[{direction}]--> Parent ID: {parent_id} ('{parent_text}')"
            )
    print("\nRoot nodes in the graph:")
    root_nodes = graph_index.get_root_nodes()
    for root in root_nodes:
        print(f"Root Node ID: {root.id}, Text: '{root.text}'")
    print("\nValidating tree structure...")
    is_valid_tree = graph_index.validate_tree()
    print(f"Is the graph a valid tree? {is_valid_tree}")
    print("-" * 50)

Sentence ID: 0, Sentence Span: (0, 42), Token Order: [1, 2, 3, 4, 5, 6, 7]
Nodes in the graph:
Node ID: 1, Text: 'Ta', UPOS: P, XPOS: P, Deprel: nsubj, Head: 2
Node ID: 2, Text: 'andis', UPOS: V, XPOS: V, Deprel: root, Head: 0
Node ID: 3, Text: 'lendurist', UPOS: S, XPOS: S, Deprel: nmod, Head: 4
Node ID: 4, Text: 'abikaasale', UPOS: S, XPOS: S, Deprel: obj, Head: 2
Node ID: 5, Text: 'oma', UPOS: P, XPOS: P, Deprel: nmod, Head: 6
Node ID: 6, Text: 'raamatu', UPOS: S, XPOS: S, Deprel: obj, Head: 2
Node ID: 7, Text: '.', UPOS: Z, XPOS: Z, Deprel: punct, Head: 2

Edges in the graph:
Node ID: 1 ('Ta') --[up]--> Parent ID: 2 ('andis')
Node ID: 2 ('andis') --[down]--> Parent ID: 1 ('Ta')
Node ID: 3 ('lendurist') --[up]--> Parent ID: 4 ('abikaasale')
Node ID: 4 ('abikaasale') --[down]--> Parent ID: 3 ('lendurist')
Node ID: 4 ('abikaasale') --[up]--> Parent ID: 2 ('andis')
Node ID: 2 ('andis') --[down]--> Parent ID: 4 ('abikaasale')
Node ID: 5 ('oma') --[up]--> Parent ID: 6 ('raamatu')
Node ID

In [10]:
# List files in the processed directory of ENC2017 of Vabamorf
vabamorf_files = glob.glob(
    str(ENC2017_DIRS["processed"] / "_vabamorf_morph_texts_json" / "*.json")
)
phrases = []
for i, filepath in tqdm.tqdm(enumerate(vabamorf_files), total=len(vabamorf_files)):
    with open(filepath, "r", encoding="utf-8") as f:
        text_obj = estnltk.converters.json_to_text(file=filepath)
        text_obj.tag_layer("morph_extended")
        stanza_syntax_tagger.tag(text_obj)
        for ss in text_obj.stanza_syntax:
            if ss.children and (ss.upostag not in ["V"] or ss.xpostag not in ["V"]):
                for child in ss.children:
                    if child.deprel in ["advmod", "nmod"] and (
                        child.upostag == "S" or child.xpostag == "S"
                    ):
                        # print(f"{ss.text} -> {child.text} (nsubj)")
                        print(f"Phrase: {child.text} {ss.text}")
                        phrases.append((child.text, ss.text))
    if i > 10:
        break

  0%|          | 2/18486 [00:00<46:53,  6.57it/s]  

Phrase: lennujaama Avraham
Phrase: lennukis koht
Phrase: Kauba koguväärtus
Phrase: väärisesemete tootenäidistega
Phrase: tehase tootenäidistega
Phrase: juhtumi uurimiseks
Phrase: kriminaalmenetlust salakaubaveo
Phrase: salakaubaveo tunnustel
Phrase: uudiste edastamise
Phrase: esmaspäeva hommikul


  0%|          | 5/18486 [00:00<48:31,  6.35it/s]  

Phrase: aasta oktoobris
Phrase: oktoobris politseinikke
Phrase: sundtoomise määruse
Phrase: kaevanduses elektrivedurist
Phrase: kaevanduses elektrilukksepana
Phrase: elektriveduri kõrvalistmel
Phrase: vagunisse sisenemiseks
Phrase: tulekahjust garaažis
Phrase: elumaja keldrist
Phrase: tänava hoovis


  0%|          | 9/18486 [00:01<31:46,  9.69it/s]

Phrase: ühisgümnaasiumi algklasside
Phrase: algklasside õpilastest
Phrase: haiguse kolmandikku
Phrase: nädala lõpul
Phrase: kuriteo koosseisu
Phrase: koosseisu puudumise
Phrase: sadama juhina
Phrase: töötajate omavalitsuse
Phrase: linnapea palka
Phrase: kuus krooni
Phrase: Teatajale palganumbrit
Phrase: linnapea palk
Phrase: maakonna linnajuhtide
Phrase: linnajuhtide töötasude
Phrase: pingereas positsioonil
Phrase: kuus krooni


  0%|          | 11/18486 [00:01<38:56,  7.91it/s]

Phrase: narkokuritegude talitus
Phrase: narkootikumide müügiga
Phrase: politseile loa
Phrase: aasta Maidu
Phrase: kuuks vahi
Phrase: teisipäeval läbiotsimise
Phrase: Lepiku tänaval
Phrase: sulguriga kilekotte
Phrase: kooli territooriumil
Phrase: territooriumil suitsetamisega
Phrase: Kullamaa valla
Phrase: valla konstaabel
Phrase: krooni trahvi
Phrase: Direktori kt
Phrase: kooliõue osas
Phrase: puuetega inimestele
Phrase: puuetega inimesed
Phrase: puuetega inimesed


In [15]:
# Save the phrases to a txt file
with open(OUTPUT_DIR / "phrases.txt", "w", encoding="utf-8") as f:
    for phrase in phrases:
        f.write(f"{phrase[0]} {phrase[1]}\n")